In [ ]:
import pandas as pd

domain = "cloud"

passage_level_corpus = pd.read_json(f"/dccstor/srosent3/mtrag_corpus_passages/{domain}-elser-index.json").transpose()

# dump in jsonl format: _id, title, text, metadata
passage_level_corpus.columns

Index(['productId', 'deliverableLoio', 'filePath', 'title', 'url', 'app_name',
       'id', 'text'],
      dtype='object')

In [25]:
passage_level_corpus = passage_level_corpus[['id','text','title']]
passage_level_corpus = passage_level_corpus.rename(columns={'id': '_id'})
passage_level_corpus.to_json(f"/dccstor/srosent3/mtrag_corpus_passages/{domain}-elser-index.jsonl", lines=True, orient='records')

In [16]:
import pandas as pd

index_name = "mt-rag-clapnq-elser-512-100-20240503"
# index_name = "mt-rag-govt-elser-512-100-20240611"
# index_name = "mt-rag-fiqa-beir-elser-512-100-20240501"
# index_name = "mt-rag-ibmcloud-elser-512-100-20240502"

domain = "clapnq"

passage_level_corpus = pd.read_json(f"/dccstor/srosent3/mtrag_corpus_passages/elastic_dump/{index_name}.jsonl.zip", lines=True)
# dump in jsonl format: _id, title, text, metadata
print(passage_level_corpus.columns)
passage_level_corpus.head(2)


Index(['_index', '_id', '_score', '_source', 'fields'], dtype='object')


,_index,_id,_score,_source,fields
0,mt-rag-clapnq-elser-512-100-20240503,837799097_6931-7548-0-617,1,"{'app_name': '', 'deliverableLoio': '', 'produ...","{'title': ['French Revolution'], 'app_name': [..."
1,mt-rag-clapnq-elser-512-100-20240503,837799097_7549-7959-0-406,1,"{'app_name': '', 'deliverableLoio': '', 'produ...","{'title': ['French Revolution'], 'app_name': [..."


In [17]:
print(len(passage_level_corpus['_id'].unique()))
print(len(passage_level_corpus))

183408
183408


In [16]:
source_data = pd.json_normalize(passage_level_corpus['_source'])

In [17]:
source_data = source_data[['id','url','text','title']]
print(source_data.columns)

Index(['id', 'url', 'text', 'title'], dtype='object')


In [18]:
print(source_data.iloc[0])
print(passage_level_corpus.iloc[0])

id                                      ibmcld_00422-0-387
url      https://cloud.ibm.com/docs/CDN?topic=CDN-set-u...
text     \n\n\n\n\n\n\n  Video - Setting up a web app w...
title                                                     
Name: 0, dtype: object
_index                mt-rag-ibmcloud-elser-512-100-20240502
_id                                       ibmcld_00422-0-387
_score                                                     1
_source    {'app_name': '', 'deliverableLoio': '', 'produ...
fields     {'title': [''], 'app_name': [''], 'text': ['

...
Name: 0, dtype: object


In [19]:
data = pd.concat([passage_level_corpus['_id'],source_data], axis=1)
print(domain)
print(len(data))
print(len(source_data))
print(len(passage_level_corpus))
# print(data.iloc[0])
data.to_json(f"/dccstor/srosent3/mtrag_corpus_passages/final/{index_name}.jsonl.zip", orient='records', lines=True)

cloud
72442
72442
72442


In [2]:
dev_tsv = pd.read_csv(f"/dccstor/srosent1/human_ai_eval/mt-rag-benchmark/human/retrieval_tasks/{domain}/qrels/dev.tsv", delimiter="\t")
print(len(dev_tsv))
dev_tsv.head(2)

494


,query-id,corpus-id,score
0,e1883ed3-18c7-46b7-aa22-3ecbac1cdcf0_1724856842,ibmcld_00474-7885-8455,1
1,e1883ed3-18c7-46b7-aa22-3ecbac1cdcf0_1724856842,ibmcld_00513-7-2197,1


In [3]:
dev_tsv[~dev_tsv['corpus-id'].isin(passage_level_corpus['_id'])]

,query-id,corpus-id,score


In [10]:
for i, row in dev_tsv[~dev_tsv['corpus-id'].isin(passage_level_corpus['_id'])].iterrows():
    print(row['corpus-id'])
    # print(row['corpus-id'][:row['corpus-id'].rindex('-', 0, row['corpus-id'].rindex('-'))])
    print(passage_level_corpus[passage_level_corpus['_id'].str.startswith(row['corpus-id'][:row['corpus-id'].rindex('-', 0, row['corpus-id'].rindex('-'))])]['_id'])